In [1]:
#%pip install requests
#%pip install beautifulsoup4
#%pip install sentence-transformers
#%pip install faiss-cpu
#%pip install transformers
#%pip install torch
#%pip install pandas
#%pip install numpy
#%pip install streamlit
#%pip install pypdf

In [2]:
import requests
from bs4 import BeautifulSoup

import numpy as np
import pandas as pd

import faiss

from pypdf import PdfReader

from io import BytesIO
from sentence_transformers import SentenceTransformer

from transformers import pipeline

import streamlit as st

m:\Coding\RAG_Pipeline_PrivacyPolicy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def extract_text_from_url(url):
    """
    Downloads webpage and extracts visible text.
    """

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0"
            )
        },
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove unwanted elements
    for tag in soup([
        "script",
        "style",
        "header",
        "footer",
        "nav"
    ]):
        tag.decompose()

    text = soup.get_text(
        separator="\n",
        strip=True
    )

    return text

In [4]:
def extract_text_from_pdf(uploaded_file):
    """
    Extract text from uploaded PDF.
    """

    pdf = PdfReader(
        BytesIO(uploaded_file.read())
    )

    text = ""

    for page in pdf.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [5]:
import re


def clean_text(text):

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove multiple newlines
    text = re.sub(
        r"\n+",
        "\n",
        text
    )

    # Remove tabs
    text = text.replace(
        "\t",
        " "
    )

    return text.strip()

In [6]:
def load_policy(source_type,
                source):

    if source_type == "url":

        raw_text = extract_text_from_url(
            source
        )

    elif source_type == "pdf":

        raw_text = extract_text_from_pdf(
            source
        )

    else:

        raise ValueError(
            "Unsupported source type"
        )

    cleaned_text = clean_text(
        raw_text
    )

    return cleaned_text

In [7]:
policy_text = load_policy(
    source_type="url",
    source="https://policies.google.com/privacy?hl=en-US"
)

print(policy_text[:1000])

Privacy Policy – Privacy & Terms – Google Skip to main content Privacy & Terms Privacy & Terms Privacy & Terms Overview Privacy Policy Terms of Service Technologies FAQ Google Account Privacy Policy Introduction Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related privacy practices Data transfer frameworks Key terms Partners Updates Google Privacy Policy When you use our services, you’re trusting us with your information. We understand this is a big responsibility and work hard to protect your information and put you in control. This Privacy Policy is meant to help you understand what information we collect, why we collect it, and how you can update, manage, export, and delete your information. If European Union or United Kingdom data protection law applies

In [8]:
print(
    f"Characters: {len(policy_text)}"
)

print(
    f"Words: {len(policy_text.split())}"
)

Characters: 89751
Words: 14439


In [9]:
def chunk_text(
    text,
    chunk_size=250,
    overlap=50
):
    """
    Split text into overlapping word chunks.

    Parameters:
        text (str)
        chunk_size (int)
        overlap (int)

    Returns:
        List[str]
    """

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += (
            chunk_size - overlap
        )

    return chunks

In [10]:
chunks = chunk_text(
    policy_text,
    chunk_size=250,
    overlap=50
)

print(f"Total chunks: {len(chunks)}")

Total chunks: 73


In [11]:
def create_chunk_metadata(
    chunks
):

    chunk_data = []

    for idx, chunk in enumerate(
        chunks
    ):

        chunk_data.append(
            {
                "chunk_id": idx,
                "text": chunk
            }
        )

    return chunk_data


In [13]:
for chunk in chunks[:5]:

    print("=" * 50)

    print(chunk[:500])

    print("\n")

chunk_data = create_chunk_metadata(chunks)

Privacy Policy – Privacy & Terms – Google Skip to main content Privacy & Terms Privacy & Terms Privacy & Terms Overview Privacy Policy Terms of Service Technologies FAQ Google Account Privacy Policy Introduction Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related pr


Information Google collects Why Google collects data Your privacy controls Sharing your information Keeping your information secure Exporting & deleting your information Retaining your information Compliance & cooperation with regulators European requirements About this policy Related privacy practices We build a range of services that help millions of people daily to explore and interact with the world in new ways. Our services include: Google apps, sites, and devices, like Search, YouTube, 